In [1]:
import sys
import os

%load_ext autoreload
%autoreload 2

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)

for p in list(sys.path):
    if 'recurrent-memory-transformer' in p:
        sys.path.remove(p)

from transformers import AutoModel, AutoTokenizer
from rl.bert_predictor import BertPredictor
from rl.text_env import TextEnv, TextReplayBuffer
from rl import WordsCounterEnv
from rl.bert_predictor import BertPredictor
from datasets import load_dataset


bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")
predictor = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

add repository dir: /home/nazar/projects/multi-step-retrieval-rl


/home/nazar/torchenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env = TextEnv()

env.embed_tokenizer = tokenizer
env.embedder = predictor


s0 = env.reset("qq", [f"t{i}"  for i in range(10)])
s0.item_ids, s0.available_ids, s0.text

([], {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'qq')

In [3]:
s1, a1, done = env.step(1)
s1.item_ids, s1.available_ids, s1.text

([1], {0, 2, 3, 4, 5, 6, 7, 8, 9}, 'qq [SEP] t1')

In [4]:
s2, a2, done = env.step(2)
s2.item_ids, s2.available_ids, s2.text

([1, 2], {0, 3, 4, 5, 6, 7, 8, 9}, 'qq [SEP] t1 [SEP] t2')

In [5]:
buffer = TextReplayBuffer(1000_000, tokenizer = tokenizer)

In [6]:
from tqdm import tqdm

for t in tqdm(range(1000)):

    s = env.reset(f"qq{t}", [f"t_{t}_{i}"  for i in range(t % 10 + 5)])

    for i in range(5):
        s_next, a_data, done = env.step(i)
        buffer.add(s, a_data, s_next, 0, done)
        s = s_next

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:03<00:00, 310.89it/s]


In [8]:
s_batch, a_batch, next_s_batch, r_batch, not_done_batch = buffer.sample(12)

In [10]:
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
print("loading dataset: AIRI-NLP/quality_counter_new_1024")
dataset = load_dataset("AIRI-NLP/quality_counter_new_1024")

loading dataset: AIRI-NLP/quality_counter_new_1024


In [11]:
predictor = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

env = WordsCounterEnv(
    dataset, 
    block_size=32, 
    embedder=predictor, 
    max_length=1024, 
    embed_tokenizer=tokenizer, 
    max_steps_count=5
)

In [12]:
buffer = TextReplayBuffer(1000_000, tokenizer = tokenizer)

In [13]:
from tqdm import tqdm

for _ in tqdm(range(10_000)):

    s = env.reset()

    for i in range(5):
        s_next, memory_block, reward, done = env.step(i)
        buffer.add(s, memory_block, s_next, reward, done)
        s = s_next

100%|██████████| 10000/10000 [02:46<00:00, 60.12it/s]


In [18]:
s_memory, a, next_s_memory, r, not_done = buffer.sample(4)

In [19]:
s_memory

TextMemory(item_ids=[[0], [], [0, 1, 2, 3], [0, 1, 2]], available_ids=[{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31}, {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31}, {4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31}, {3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31}], input_ids=tensor([[  101,  1293,  1242,  1551,  1103,  1937,   112,  2951,   112,  2691,
          1107,  1103,  3087,   136,   102, 10368,  9853,  5444,  1120, 13280,
          1116,  1181,  1830,   119, 15661,   168,   176, 22540,   134,   168,
           176, 22540,   197,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,   

In [20]:
a

TextMemoryItem(index=[1, 0, 4, 3], input_ids=tensor([[  101,   197,   164,   166,   132,   168,   176, 22540,   119,  4684,
           113,   164,   112,   168,  1383,  7409,  2528,  8355,   112,   117,
           112,   190,  1161,   118,  3413,  1604,  1571, 25041,  1527,   118,
           124,   112,   166,   102],
        [  101,  1293,  1242,  1551,  1103,  1937,   112,  4267, 23397,   112,
          2691,  1107,  1103,  3087,   136,   102,  2885,   117,  1105,  1131,
          1261,  1103,  1948,  1121,  1139,  1289,  1177,  1115,   178,  1225,
          1136,  1267,  1123,   102],
        [  101,   117,  1218,   119,   119,   119,   113,  8557,   114,  1177,
          1165,  1132,  1128,  1280,  1106,  1243,  1240,  2261,   136,  4035,
          2386,   178,   112,   182,  1684,  1113,  1122,   119,   119,   119,
           178,   112,  1396,   102],
        [  101,  5830,   119,  2561, 11194,  1880,   113,   112,  5444,   112,
           114,   132,   176,  1161,   119,  2076, 

In [9]:
embeds.shape, r.shape, not_done.shape 

(torch.Size([4, 32, 256]), torch.Size([4, 1]), torch.Size([4, 1]))